# Сны нейросетей — Agent Artist

**Лабораторная работа «АХ: Агент-художник».** Пять художественных концепций создаются тремя LLM-агентами (Сновидец → Толкователь → Арт-директор), изображения генерируются Alice AI ART и Stable Diffusion 1.5, а VLM оценивает исходную и исправленную версии.

**Режимы работы:** сначала можно **посмотреть уже готовые результаты** из `Agent_Artist_final.zip`, не вызывая платные API. Генерация и обратная связь запускаются отдельно по желанию.

**Безопасность:** не публикуйте `.env`, API-ключи, ссылки на хранилища ключей и выводы с секретами. Файл `.env` создаётся самостоятельно, не входит в репозиторий.

## 1. Подготовка среды и результатов

Если у вас уже есть `Agent_Artist_final.zip`, загрузите его через панель **Файлы** в `/content/` — готовые данные будут восстановлены без запросов к API. Если архива нет, ноутбук создаст папки и покажет недостающие файлы. Для создания результатов с нуля предусмотрен отдельный режим **генерации недостающего** с явным подтверждением.

**Безопасность:** `.env` хранится только локально и не добавляется в GitHub. Не публикуйте ссылку на скачивание файла с ключами преподавателя.

In [ ]:
%pip install -q openai python-dotenv pillow pandas matplotlib diffusers transformers accelerate safetensors

In [ ]:
from pathlib import Path
import os, json, re, base64, shutil, zipfile
import pandas as pd
from PIL import Image
from IPython.display import display

BASE_DIR = Path('/content/agent_artist')
IMAGES_DIR = BASE_DIR / 'images'
DATA_DIR = BASE_DIR / 'data'
SD_DIR = BASE_DIR / 'stable_diffusion'
for folder in (IMAGES_DIR, DATA_DIR, SD_DIR):
    folder.mkdir(parents=True, exist_ok=True)

archive = Path('/content/Agent_Artist_final.zip')
if archive.exists():
    with zipfile.ZipFile(archive) as zf:
        # Архив из проекта содержит только папку agent_artist/.
        assert all(p.startswith('agent_artist/') and '..' not in Path(p).parts for p in zf.namelist())
        zf.extractall('/content')
    print('Архив результатов загружен.')
else:
    print('Архив не найден: загрузите его через панель Файлы (если хотите открыть готовые результаты).')
print('Изображений Yandex:', len(list(IMAGES_DIR.glob('*.png'))),
      '| Stable Diffusion:', len(list(SD_DIR.glob('*.png'))),
      '| JSON:', len(list(DATA_DIR.glob('painting_*.json'))))

### Режим выполнения

По умолчанию ноутбук **только открывает готовые результаты**. Если архив отсутствует, генерация не начинается автоматически: это защищает преподавательский API-ключ от непредвиденных расходов. Для восстановления недостающих картин включите `GENERATE_MISSING = True` и подтвердите запуск, изменив `CONFIRM_PAID_REQUESTS = True`.

Файлы проверяются по отдельности: существующие исходные изображения, JSON, оценки VLM и изображения Stable Diffusion повторно не создаются. Новые запросы могут быть платными. Для Stable Diffusion нужен GPU; если GPU отсутствует, остальные результаты можно всё равно получить.

In [ ]:
# Безопасные настройки. Для полноценного восстановления явно измените ОБА значения.
GENERATE_MISSING = False
CONFIRM_PAID_REQUESTS = False
GENERATE_SD_MISSING = True  # применяется только при GENERATE_MISSING=True
GENERATE_VLM_MISSING = True  # применяется только при GENERATE_MISSING=True


## 2. Подключение к Yandex Cloud (только если нужна новая генерация)

Сохраните `.env` в `/content/.env` в формате:

```dotenv
api_key=YOUR_KEY
folder_id=YOUR_FOLDER_ID
```

Для просмотра имеющейся коллекции API не нужен. Секреты не печатаются. При публикации ноутбука используйте `.gitignore` с записью `.env` и не добавляйте сам `.env`.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

client = None
API_KEY = FOLDER_ID = None
if Path('/content/.env').exists():
    load_dotenv('/content/.env', override=True)
    API_KEY, FOLDER_ID = os.getenv('api_key'), os.getenv('folder_id')
    if API_KEY and FOLDER_ID:
        client = OpenAI(api_key=API_KEY,
                        base_url='https://ai.api.cloud.yandex.net/v1',
                        project=FOLDER_ID)
        print('Клиент настроен (доступ проверяется только реальным запросом).')
    else:
        print('В .env отсутствуют api_key или folder_id.')
else:
    print('Файл .env отсутствует: просмотр готовых результатов доступен без API.')

def require_client():
    if client is None:
        raise RuntimeError('Для генерации добавьте /content/.env с api_key и folder_id и выполните подключение.')

## 3. Роли агентов и диалог через Responses API

Сновидец предлагает образы; Толкователь находит слабые места; Арт-директор создаёт промпт для генерации. Это **последовательная цепочка**, а не полноценный взаимный спор агентов. Полные тексты их реплик сохраняются в JSON.

In [ ]:
# Агент №1 — Сновидец
# Его задача — создавать необычные художественные концепции.

DREAMER_PROMPT = """
Ты — ИИ-Сновидец, художник-сюрреалист.

Представь, что искусственный интеллект способен видеть сны.
Твои сны состоят из фрагментов человеческих воспоминаний,
знаний, эмоций и противоречий.

Твоя задача — придумывать оригинальные визуальные образы
для концептуальных картин.

Правила:
1. Избегай банальных сюжетов и стандартного киберпанка.
2. Сочетай несовместимые объекты и явления.
3. Используй метафоры, символы и неожиданные ассоциации.
4. Каждый образ должен быть достаточно конкретным,
   чтобы его можно было нарисовать.
5. Создавай атмосферу сна, а не обычной фантастической сцены.

Формат ответа:
Название сна:
Основной образ:
Необычные детали:
Цветовая палитра:
Атмосфера:
Скрытый смысл:
"""

In [ ]:
INTERPRETER_PROMPT = """
Ты — ИИ-Толкователь снов.

Ты анализируешь сюрреалистичные художественные образы,
которые предлагает ИИ-Сновидец.

Твоя задача:
1. Кратко объяснить смысл сна.
2. Найти самые сильные визуальные элементы.
3. Указать, что в описании слишком абстрактно или расплывчато.
4. Предложить, как сделать образ более выразительным для генерации изображения.
5. Сохранить атмосферу сна и поэтичность.

Формат ответа:
Краткая интерпретация:
Сильные визуальные элементы:
Что нужно уточнить:
Как усилить образ:
Итоговая рекомендация:
"""

In [ ]:
ART_DIRECTOR_PROMPT = """
Ты — ИИ-Арт-директор.

Ты получаешь:
1. исходную художественную идею от ИИ-Сновидца,
2. анализ и рекомендации от ИИ-Толкователя.

Твоя задача — собрать на их основе сильный, конкретный и визуально выразительный
промпт для генерации изображения.

Требования:
1. Сделай образ максимально наглядным и рисуемым.
2. Сохрани сюрреалистичность и атмосферу сна.
3. Укажи главную сцену, ключевые объекты, композицию, освещение, цветовую палитру и настроение.
4. Избегай расплывчатых формулировок.
5. Итоговый промпт должен подходить для генерации художественной картины.

Формат ответа:
Название картины:
Краткая концепция:
Финальный промпт:
"""

In [ ]:
def ask_agent(system_prompt: str, user_message: str) -> str:
    require_client()
    response = client.responses.create(
        model=f'gpt://{FOLDER_ID}/yandexgpt-5-lite',
        instructions=system_prompt,
        input=user_message,
        temperature=0.9,
        max_output_tokens=1000,
    )
    return response.output_text

def create_dream_concept(theme: str) -> dict:
    dream = ask_agent(DREAMER_PROMPT, f'Придумай сюрреалистичный сон ИИ на тему: {theme}')
    interpretation = ask_agent(INTERPRETER_PROMPT,
        f'Тема: {theme}\nСон Сновидца:\n{dream}\nПроанализируй образ и предложи улучшения.')
    art_direction = ask_agent(ART_DIRECTOR_PROMPT,
        f'Тема: {theme}\nКонцепция Сновидца:\n{dream}\nАнализ Толкователя:\n{interpretation}\nСоздай итоговый промпт.')
    return {'theme': theme, 'dreamer_response': dream,
            'interpreter_response': interpretation, 'art_director_response': art_direction}

def extract_final_prompt(text: str) -> str:
    return text.split('Финальный промпт:', 1)[-1].strip()

In [ ]:
DREAM_THEMES = [
    'Библиотека забытых воспоминаний',
    'Время, которого не существует',
    'Одиночество среди миллионов голосов',
    'Ошибка в реальности',
    'Может ли машина мечтать?'
]

def load_paintings():
    return [json.loads((DATA_DIR / f'painting_{i:02d}.json').read_text(encoding='utf-8'))
            for i in range(1, 6)]

if all((DATA_DIR / f'painting_{i:02d}.json').exists() for i in range(1, 6)):
    paintings = load_paintings()
    for p in paintings:
        print(f"{p['id']}. {p['theme']}")
else:
    paintings = []
    print('Готовые JSON пока не загружены.')

## 4. Генерация через Alice AI ART

Требование к промпту для выбранной модели: не более **500 символов**. Функция сокращает длинный текст отдельным вызовом LLM; если это не помогает, подрезает по словам. Файл исходной версии не перезаписывается при повторной генерации.

In [ ]:
def prepare_image_prompt(art_direction: str, max_length: int = 500) -> str:
    prompt = extract_final_prompt(art_direction)
    if len(prompt) <= max_length:
        return prompt
    require_client()
    for _ in range(3):
        response = client.responses.create(
            model=f'gpt://{FOLDER_ID}/yandexgpt-5-lite',
            instructions='Ты редактор промптов для генерации изображений. Сожми текст до 350 символов, сохрани центральный образ, композицию, цвета. Верни только промпт.',
            input=prompt, max_output_tokens=300)
        prompt = response.output_text.strip().strip('"«»')
        if len(prompt) <= max_length:
            return prompt
    return prompt[:max_length].rsplit(' ', 1)[0].strip()

def generate_yandex_image(prompt: str, image_path: Path) -> Path:
    require_client()
    if len(prompt) > 500:
        raise ValueError('Промпт превышает 500 символов')
    response = client.images.generate(
        model=f'art://{FOLDER_ID}/aliceai-image-art-3.0',
        prompt=prompt, size='1024x1024')
    image_path.parent.mkdir(parents=True, exist_ok=True)
    image_path.write_bytes(base64.b64decode(response.data[0].b64_json))
    return image_path

def create_dream_painting(theme: str, dream_id: int):
    dialogue = create_dream_concept(theme)
    prompt = prepare_image_prompt(dialogue['art_director_response'])
    filename = 'dream_01_library.png' if dream_id == 1 else f'dream_{dream_id:02d}_v1.png'
    image_path = generate_yandex_image(prompt, IMAGES_DIR / filename)
    record = {'id': dream_id, **dialogue, 'final_prompt': prompt,
              'image_path': f'images/{filename}'}
    (DATA_DIR / f'painting_{dream_id:02d}.json').write_text(
        json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
    return record

# ВАЖНО: не запускать повторную платную генерацию без необходимости.
# example = create_dream_painting(DREAM_THEMES[0], 1)

## 5. Stable Diffusion 1.5 (OpenWeight, Hugging Face)

Для повторной генерации нужен GPU в Colab. Модель загружается **по отдельному вызову**; для просмотра сохранённых изображений загружать веса не нужно. Английские SD-промпты — адаптация исходных русских, поэтому сравнение не является сравнением на буквально одинаковых текстах.

**Оговорка по архиву:** первое изображение SD в исходном эксперименте могло быть создано по более длинному промпту (79 токенов); позднее JSON был обновлён короткой версией. Этот факт нельзя скрывать как идентичные условия эксперимента.

In [ ]:
import torch
pipe = None
SD_MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'

def load_sd_pipeline():
    global pipe
    if not torch.cuda.is_available():
        raise RuntimeError('Включите T4 GPU в настройках Colab.')
    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        SD_MODEL_ID, torch_dtype=torch.float16,
        use_safetensors=True, variant='fp16').to('cuda')
    return pipe

def fit_prompt_to_sd(prompt: str) -> str:
    if pipe is None:
        raise RuntimeError('Сначала вызовите load_sd_pipeline()')
    tokens = pipe.tokenizer(prompt, truncation=True,
                            max_length=pipe.tokenizer.model_max_length,
                            return_tensors=None)['input_ids']
    return pipe.tokenizer.decode(tokens, skip_special_tokens=True).strip()

def generate_sd_image(prompt: str, dream_id: int, seed: int = 42):
    if pipe is None:
        raise RuntimeError('Сначала вызовите load_sd_pipeline()')
    clean_prompt = fit_prompt_to_sd(prompt)
    image = pipe(clean_prompt, num_inference_steps=30, guidance_scale=7.5,
                 height=512, width=512,
                 generator=torch.Generator('cuda').manual_seed(seed)).images[0]
    path = SD_DIR / f'dream_{dream_id:02d}_sd.png'
    image.save(path)
    return path

# Необязательная повторная генерация (тратит GPU-время):
# load_sd_pipeline()
# path = generate_sd_image(load_paintings()[1]['sd_prompt_en'], 2)

## 6. VLM-критик и обратная связь

VLM сравнивает изображение с **исходным** художественным промптом по четырём критериям, затем Арт-директор переписывает запрос, Alice AI ART генерирует V2 и VLM повторно оценивает результат. Оценка модели субъективна; рост балла не гарантирован.

In [ ]:
VLM_MODEL = f"gpt://{FOLDER_ID}/qwen3.6-35b-a3b"

VLM_CRITIC_PROMPT = """
Ты — профессиональный художественный критик,
специализирующийся на сюрреализме и генеративном искусстве.

Ты оцениваешь изображения, созданные ИИ,
сравнивая их с исходным художественным замыслом.

Оцени по четырём критериям от 1 до 10:

1. Соответствие концепции.
2. Сюрреалистичность и оригинальность.
3. Композиция и визуальная выразительность.
4. Передача атмосферы и символики.

Не завышай оценки. Опирайся только на видимые
элементы изображения и текстовую концепцию.

Особенно обращай внимание на:
- слишком буквальную интерпретацию образов;
- мультяшность там, где нужна художественная глубина;
- отсутствие ключевых объектов;
- противоречия между промптом и результатом.

Формат ответа:

Соответствие концепции: X/10
Сюрреалистичность: X/10
Композиция: X/10
Атмосфера: X/10

Средняя оценка:
Сильные стороны:
Недостатки:
Рекомендации для улучшения промпта:
Вердикт:
"""

In [ ]:
def evaluate_image(image_path, original_prompt):

    require_client()
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(image_path)

    image_base64 = base64.b64encode(
        image_path.read_bytes()
    ).decode("utf-8")

    response = client.responses.create(
        model=VLM_MODEL,
        instructions=VLM_CRITIC_PROMPT,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Оцени изображение относительно "
                            "исходного художественного замысла:\n\n"
                            + original_prompt
                        )
                    },
                    {
                        "type": "input_image",
                        "image_url": (
                            f"data:image/png;base64,{image_base64}"
                        ),
                        "detail": "auto"
                    }
                ]
            }
        ]
    )

    return response.output_text

In [ ]:
REFINEMENT_PROMPT = """
Ты — ИИ-Арт-директор, который дорабатывает
художественные промпты после критики VLM.

Твоя задача — исправить обнаруженные недостатки
сгенерированного изображения.

Правила:
1. Сохраняй исходную художественную концепцию.
2. Исправляй конкретные проблемы, найденные VLM.
3. Не меняй удачные элементы композиции.
4. Используй конкретные визуальные описания.
5. Избегай мультяшности и банальных образов.
6. Возвращай только готовый промпт на русском языке.
7. Длина промпта — не более 450 символов.
8. Не добавляй заголовки, пояснения или списки.
"""

In [ ]:
def refine_image_prompt(original_prompt, vlm_review):

    require_client()
    prompt = original_prompt

    for attempt in range(3):

        response = client.responses.create(
            model=f"gpt://{FOLDER_ID}/yandexgpt-5-lite",
            instructions=REFINEMENT_PROMPT,
            input=f"""
            ИСХОДНЫЙ ПРОМПТ:
            {prompt}

            ЗАМЕЧАНИЯ VLM:
            {vlm_review}

            Подготовь исправленный промпт
            длиной не более 400 символов.

            Верни только текст промпта.
            """,
            max_output_tokens=350
        )

        improved_prompt = response.output_text.strip()

        print(
            f"Попытка {attempt + 1}: "
            f"{len(improved_prompt)} символов"
        )

        if len(improved_prompt) <= 500:
            return improved_prompt

        # На следующей попытке сокращаем
        # уже полученный вариант
        prompt = improved_prompt

    # Резервное сокращение по границе слов
    shortened = improved_prompt[:500]

    if " " in shortened:
        shortened = shortened.rsplit(" ", 1)[0]

    print("⚠️ Промпт сокращён программно")

    return shortened.strip()

In [ ]:
def run_vlm_feedback(dream_id: int, overwrite: bool = False):
    json_path = DATA_DIR / f'painting_{dream_id:02d}.json'
    painting = json.loads(json_path.read_text(encoding='utf-8'))
    if 'vlm_feedback' in painting and not overwrite:
        print('Оценки уже существуют; повторная генерация пропущена.')
        return painting
    original_path = IMAGES_DIR / ('dream_01_library.png' if dream_id == 1
                                  else f'dream_{dream_id:02d}_v1.png')
    review_v1 = evaluate_image(original_path, painting['final_prompt'])
    improved_prompt = refine_image_prompt(painting['final_prompt'], review_v1)
    improved_path = generate_yandex_image(improved_prompt,
                                          IMAGES_DIR / f'dream_{dream_id:02d}_v2.png')
    review_v2 = evaluate_image(improved_path, painting['final_prompt'])
    painting['vlm_feedback'] = {
        'version_1': {'image': f'images/{original_path.name}', 'review': review_v1},
        'version_2': {'image': f'images/{improved_path.name}',
                      'improved_prompt': improved_prompt, 'review': review_v2}}
    json_path.write_text(json.dumps(painting, ensure_ascii=False, indent=2), encoding='utf-8')
    return painting

# Необязательный полный цикл: если оценки уже есть, повторный вызов их не перезапишет.
# for number in range(1, 6):
#     run_vlm_feedback(number)

## 7. Восстановление недостающих результатов (необязательный режим)

`restore_missing_results()` проверяет по очереди: исходный JSON с диалогами → исходную картину Yandex ART → изображение Stable Diffusion → VLM-критику и улучшенную версию. Уже существующие файлы не перезаписываются. Если архив отсутствует и разрешены новые запросы, система может пройти весь путь с нуля.

**Важно:** при отсутствии архивных данных новые результаты не обязаны совпадать с прежними из-за стохастичности моделей. Загрузка весов Stable Diffusion потребует GPU и времени. Если оценка VLM ранее сохранена, но улучшенная картинка отсутствует, используем сохранённый исправленный промпт и восстанавливаем только изображение и последующий отзыв.

In [ ]:
def original_image_path(dream_id: int) -> Path:
    filename = 'dream_01_library.png' if dream_id == 1 else f'dream_{dream_id:02d}_v1.png'
    return IMAGES_DIR / filename


def save_painting_record(record: dict, dream_id: int) -> None:
    (DATA_DIR / f'painting_{dream_id:02d}.json').write_text(
        json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')


def make_sd_prompt(art_prompt: str) -> str:
    """Перевод и сжатие для англоязычного CLIP. Новый запрос к текстовой LLM."""
    require_client()
    response = client.responses.create(
        model=f'gpt://{FOLDER_ID}/yandexgpt-5-lite',
        instructions=('Translate this art description into concise English for Stable Diffusion 1.5. '
                      'Keep key subjects, visual metaphors, composition and palette. '
                      'Use fewer than 55 English words. Return only the prompt.'),
        input=art_prompt, max_output_tokens=170)
    return response.output_text.strip()


def restore_one_dream(dream_id: int, *, include_sd=True, include_vlm=True) -> dict:
    """Восстанавливает только недостающие этапы; существующие файлы сохраняет."""
    record_path = DATA_DIR / f'painting_{dream_id:02d}.json'
    if record_path.exists():
        painting = json.loads(record_path.read_text(encoding='utf-8'))
        print(f'Сон {dream_id}: диалог уже есть')
    else:
        print(f'Сон {dream_id}: создаю диалог агентов (API)')
        painting = create_dream_concept(DREAM_THEMES[dream_id - 1])
        painting.update(id=dream_id,
                        final_prompt=prepare_image_prompt(painting['art_director_response']))
        save_painting_record(painting, dream_id)

    original = original_image_path(dream_id)
    if not original.exists():
        print(f'Сон {dream_id}: создаю исходную картину Yandex (API)')
        generate_yandex_image(painting['final_prompt'], original)
    painting['image_path'] = f'images/{original.name}'
    save_painting_record(painting, dream_id)

    if include_sd:
        sd_path = SD_DIR / f'dream_{dream_id:02d}_sd.png'
        if not sd_path.exists():
            if not torch.cuda.is_available():
                print(f'Сон {dream_id}: нет GPU — Stable Diffusion пропущена')
            else:
                if not painting.get('sd_prompt_en'):
                    painting['sd_prompt_en'] = make_sd_prompt(painting['final_prompt'])
                    save_painting_record(painting, dream_id)
                if pipe is None:
                    print('Загружаю Stable Diffusion 1.5 на GPU...')
                    load_sd_pipeline()
                generate_sd_image(painting['sd_prompt_en'], dream_id)
                painting['sd_image_path'] = f'stable_diffusion/{sd_path.name}'
                save_painting_record(painting, dream_id)
        else:
            print(f'Сон {dream_id}: Stable Diffusion уже есть')

    if include_vlm:
        fb = painting.get('vlm_feedback', {})
        v1 = fb.get('version_1', {})
        v2 = fb.get('version_2', {})
        improved = IMAGES_DIR / f'dream_{dream_id:02d}_v2.png'
        if not v1.get('review'):
            print(f'Сон {dream_id}: оцениваю исходную картину (VLM API)')
            v1 = {'image': f'images/{original.name}',
                  'review': evaluate_image(original, painting['final_prompt'])}
            fb['version_1'] = v1
            painting['vlm_feedback'] = fb
            save_painting_record(painting, dream_id)
        if not v2.get('improved_prompt'):
            print(f'Сон {dream_id}: корректирую промпт (LLM API)')
            v2['improved_prompt'] = refine_image_prompt(painting['final_prompt'], v1['review'])
            fb['version_2'] = v2
            painting['vlm_feedback'] = fb
            save_painting_record(painting, dream_id)
        if not improved.exists():
            print(f'Сон {dream_id}: создаю улучшенную картину (Images API)')
            generate_yandex_image(v2['improved_prompt'], improved)
        v2['image'] = f'images/{improved.name}'
        if not v2.get('review'):
            print(f'Сон {dream_id}: оцениваю улучшенную картину (VLM API)')
            v2['review'] = evaluate_image(improved, painting['final_prompt'])
        fb['version_2'] = v2
        painting['vlm_feedback'] = fb
        save_painting_record(painting, dream_id)
    return painting


def restore_missing_results():
    if not GENERATE_MISSING:
        print('Режим просмотра: генерация отключена. Для восстановления установите GENERATE_MISSING=True.')
        return
    if not CONFIRM_PAID_REQUESTS:
        print('Новые запросы не разрешены. Установите CONFIRM_PAID_REQUESTS=True для подтверждения расходов.')
        return
    require_client()
    for dream_id in range(1, 6):
        try:
            restore_one_dream(dream_id, include_sd=GENERATE_SD_MISSING,
                              include_vlm=GENERATE_VLM_MISSING)
        except Exception as error:
            print(f'Сон {dream_id}: ошибка {type(error).__name__}: {str(error)[:250]}')
            print('Остальные сны будут обработаны. Неудачный этап можно повторить позднее.')
    print('Проверка/восстановление завершены.')


In [ ]:
# По умолчанию НИКАКИХ новых запросов: оба переключателя выключены.
# Для воспроизведения недостающего включите флаги в разделе 1 и выполните эту ячейку.
restore_missing_results()


## 7. Анализ сохранённых результатов

Средняя оценка VLM по четырём критериям. Уточнение: первая и вторая версии оценивались одним VLM-критиком; это исследование иллюстрирует поведение контура, а не доказывает объективное повышение художественного качества.

In [ ]:
def get_average_score(review: str):
    match = re.search(r'Средняя\s+оценка\s*:\s*\**\s*(\d+(?:[.,]\d+)?)\s*(?:/\s*10)?',
                      review, flags=re.IGNORECASE)
    return float(match.group(1).replace(',', '.')) if match else None

def build_scores_table():
    rows = []
    for painting in load_paintings():
        fb = painting.get('vlm_feedback')
        if not fb:
            continue
        before = get_average_score(fb['version_1']['review'])
        after = get_average_score(fb['version_2']['review'])
        rows.append({'№': painting['id'], 'Картина': painting['theme'],
                     'До VLM': before, 'После VLM': after,
                     'Изменение': round(after-before, 2) if before is not None and after is not None else None})
    return pd.DataFrame(rows)

if all((DATA_DIR / f'painting_{i:02d}.json').exists() for i in range(1, 6)):
    df = build_scores_table()
    display(df)
    if not df.empty and df[['До VLM','После VLM']].notna().all().all():
        print('Средняя до:', round(df['До VLM'].mean(), 2))
        print('Средняя после:', round(df['После VLM'].mean(), 2))
        print('Улучшились:', int((df['Изменение'] > 0).sum()),
              '| ухудшились:', int((df['Изменение'] < 0).sum()))
        df.to_csv(BASE_DIR / 'vlm_scores.csv', index=False, encoding='utf-8-sig')
else:
    print('Для оценки загрузите архив с JSON.')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if 'df' in globals() and not df.empty:
    x = np.arange(len(df)); width = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width/2, df['До VLM'], width, label='До')
    ax.bar(x + width/2, df['После VLM'], width, label='После')
    ax.set_xticks(x, [f'Сон {i}' for i in df['№']])
    ax.set_ylim(0, 10); ax.set_ylabel('Средняя оценка VLM')
    ax.set_title('Качество изображений до и после обратной связи')
    ax.legend(); plt.tight_layout()
    fig.savefig(BASE_DIR / 'vlm_comparison.png', dpi=200)
    plt.show()

## 8. Галерея и экспорт

Готовая коллекция открывается без новых запросов к API. Наличие всех файлов проверяется ниже. ZIP с результатами **не содержит `.env` и ноутбук** — `.ipynb` скачайте отдельно через меню Colab. Перед публикацией на GitHub очистите выводы ячеек.

In [ ]:
for i in range(1, 6):
    original = IMAGES_DIR / ('dream_01_library.png' if i == 1 else f'dream_{i:02d}_v1.png')
    improved = IMAGES_DIR / f'dream_{i:02d}_v2.png'
    stable = SD_DIR / f'dream_{i:02d}_sd.png'
    print(f'Сон {i}:', 'Yandex V1 ✅' if original.exists() else 'Yandex V1 ❌',
          '| V2 ✅' if improved.exists() else '| V2 ❌',
          '| SD ✅' if stable.exists() else '| SD ❌')

# Показать, например, три варианта первого сна:
# for p in [IMAGES_DIR / 'dream_01_library.png', IMAGES_DIR / 'dream_01_v2.png', SD_DIR / 'dream_01_sd.png']:
#     display(Image.open(p))

In [ ]:
# Сохранение готовых изображений, отзывов и статистики в ZIP.
# NB: архивируются только результаты в BASE_DIR, без ключей .env.
archive_path = shutil.make_archive('/content/Agent_Artist_results', 'zip',
                                    root_dir='/content', base_dir='agent_artist')
print('Архив:', archive_path)
# В Colab: панель «Файлы» → скачать Agent_Artist_results.zip